In [2]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
# from sklearn.manifold import TSNE
import glasbey
import os
import h5py
import numpy as np
from sklearn.decomposition import PCA
import openTSNE

pio.renderers.default = 'browser'

In [4]:
# read in matlab output
mat_filename = r"Y:\3darena_behavior\wildtype_062425\011725_9\011725_9_3d_arenah_070225\cluster_output\Combined_Results\Results\test1\session_1_out.mat"

# "C:\Users\gangliagurdian\Desktop\Mias Folder\data testing\10min testing\Combined_Results\Results\test1\session_1_out.mat"
# "Z:\socialbox_fp\C2_042125_C3_042125_gpe\C2_042125\042125_1\open_field_061625\day_1\042125_1_tdt_rt_061625of\Bonsai - Copy\AccelCluster_Output - Copy\Combined_Results\Results\test1\session_1_out.mat"

In [5]:
# extract similarity matrix & cluster labels
if os.path.exists(mat_filename):
    with h5py.File(mat_filename, 'r') as file:
        clusters = file['Clusters']
        similarity_matrix = clusters['sim'][()]

        cluster_labels = clusters["idx"][()]
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')

print(similarity_matrix.shape)
print(cluster_labels.shape)

# transpose cluster_labels
cluster_labels = cluster_labels.T

# turn clusters_labels into 1D array
cluster_labels = cluster_labels.flatten()
print(cluster_labels.shape)

(47715, 47715)
(1, 47715)
(47715,)


In [6]:
# apply PCA to reduce dimensions to 50
pca = PCA(n_components=50)
similarity_matrix_reduced = pca.fit_transform(similarity_matrix)
print(similarity_matrix_reduced.shape)

(47715, 50)


In [7]:
# use openTSNE to visualize the 50 PCA components in 2D
tsne = openTSNE.TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
similarity_matrix_tsne = tsne.fit(similarity_matrix_reduced)

: 

In [15]:
aff50 = openTSNE.affinity.PerplexityBasedNN(
    similarity_matrix_reduced,
    perplexity=50,
    n_jobs=32,
    random_state=0,
)

init = openTSNE.initialization.rescale(similarity_matrix_reduced[:, :2])

embedding_aff500 = openTSNE.TSNE(
    n_jobs=32,
    verbose=True,
).fit(affinities=aff50, initialization=init)

: 

In [12]:
# use t-SNE to visualize the 50 PCA components in 2D
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42) # perplexity 477 before
similarity_matrix_tsne = TSNE.fit(similarity_matrix_reduced)
print(similarity_matrix_tsne.shape)


cluster_labels_str = [str(c) for c in cluster_labels]
cluster_names = sorted(set(cluster_labels_str))

glasbey_palette = glasbey.create_palette(palette_size=len(cluster_names))
cluster_to_color = {
    name: glasbey_palette[i]
    for i, name in enumerate(cluster_names)
}

fig = px.scatter(
    x=similarity_matrix_tsne[:, 0],
    y=similarity_matrix_tsne[:, 1],
    color=cluster_labels_str,
    title='t-SNE Visualization of Similarity Matrix',
    color_discrete_map=cluster_to_color
)
fig.update_traces(marker=dict(size=8))
fig.show()


AttributeError: 'numpy.ndarray' object has no attribute 'verbose'